# S3 Post Pull
Pulls social post data from S3 parquet files into a flat table with columns:
`wiki_id`, `thread_id`, `post_id`, `title`, `content`, `post_time`, `user_id`, `user_name`, `n_likes`

> **Note:** `user_name` is not stored in the parquet dumps. It is generated deterministically from `user_id` using `generate_username()`, so the same user always gets the same pseudonym.


## 1. Setup

In [1]:
import boto3
import duckdb
import pandas as pd

BUCKET_PREFIX = "s3://de-data-science-experiments/social_content_data"

session = boto3.Session()
creds = session.get_credentials().get_frozen_credentials()

# Bucket region (recommended)
s3 = session.client("s3")
REGION = session.region_name or "us-east-1"

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"SET s3_region='{REGION}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")
con.execute(f"SET s3_session_token='{creds.token}';")

print(f"Connected — region: {REGION}")
print(f"Access key prefix: {creds.access_key[:8]}...")


Connected — region: us-east-1
Access key prefix: ASIASPF5...


In [2]:
import random
import hashlib

_ADJECTIVES = [
    "Swift", "Brave", "Calm", "Dark", "Eager", "Faint", "Glad", "Harsh",
    "Idle", "Jolly", "Kind", "Lofty", "Mild", "Noble", "Odd", "Proud",
    "Quick", "Rare", "Shy", "Tidy", "Utter", "Vast", "Warm", "Witty",
    "Young", "Zany", "Bold", "Cool", "Deft", "Epic",
]

_NOUNS = [
    "Fox", "Wolf", "Bear", "Hawk", "Lynx", "Crow", "Drake", "Fawn",
    "Hare", "Ibis", "Jade", "Kite", "Lark", "Mink", "Newt", "Orca",
    "Pike", "Quail", "Rook", "Sage", "Teal", "Vole", "Wren", "Yak",
    "Zeal", "Apex", "Bard", "Cove", "Dune", "Echo",
]


def generate_username(user_id: int | None = None, *, seed: int | None = None) -> str:
    """
    Generate a deterministic random username.

    - If `user_id` is provided, the same ID always produces the same name.
    - If `seed` is provided instead, that seed controls randomness.
    - If neither is given, a fully random name is returned.

    Format: <Adjective><Noun><4-digit number>  e.g. "SwiftFox4821"
    """
    if user_id is not None:
        # Derive a stable integer seed from the user_id via hashing
        digest = hashlib.md5(str(user_id).encode()).digest()
        rng_seed = int.from_bytes(digest[:8], "little")
    elif seed is not None:
        rng_seed = seed
    else:
        rng_seed = None  # truly random

    rng = random.Random(rng_seed)
    adj    = rng.choice(_ADJECTIVES)
    noun   = rng.choice(_NOUNS)
    number = rng.randint(1000, 9999)
    return f"{adj}{noun}{number}"


# Quick sanity check
print(generate_username(user_id=42))    # deterministic
print(generate_username(user_id=42))    # same output
print(generate_username(user_id=99))    # different user → different name
print(generate_username())              # random each call


BoldDrake7986
BoldDrake7986
SwiftIbis5563
ProudCrow1962


## 2. Configuration
Set the list of wiki IDs to pull. Set `WIKI_IDS = None` to pull all wikis in the catalogue.

In [3]:
# Wiki catalogue for reference
WIKI_CATALOGUE = [
    {"wiki_id": 2051849, "name": "Dragon Adventures Wiki"},
    {"wiki_id": 3666731, "name": "Roblox Grow a Garden Wiki"},
    {"wiki_id": 1619358, "name": "Adopt Me Wiki"},
    {"wiki_id": 3593972, "name": "FORSAKEN Wiki"},
    {"wiki_id": 3454593, "name": "Dandy's World Wiki"},
    {"wiki_id": 2308060, "name": "Creatures of Agartha Wiki"},
    {"wiki_id": 3651579, "name": "Die of Death Wiki"},
    {"wiki_id": 599406,  "name": "My Singing Monsters Wiki"},
    {"wiki_id": 997342,  "name": "Boku no Hero Academia Wiki"},
    {"wiki_id": 1081,    "name": "One Piece"},
    {"wiki_id": 1318,    "name": "Narutopedia"},
    {"wiki_id": 2169014, "name": "Project SEKAI Wiki"},
    {"wiki_id": 249799,  "name": "Battle for Dream Island Wiki"},
    {"wiki_id": 374,     "name": "Disney Wiki"},
    {"wiki_id": 2860,    "name": "Encyclopedia SpongeBobia"},
    {"wiki_id": 35171,   "name": "Hunger Games"},
    {"wiki_id": 2233,    "name": "Marvel"},
    {"wiki_id": 147,     "name": "Star Wars"},
    {"wiki_id": 509,     "name": "Harry Potter"},
]

# Set to a list of wiki IDs to restrict, or None for all
WIKI_IDS = [35171, 3454593]  # e.g. [509, 1081] or None

target_wikis = (
    WIKI_CATALOGUE if WIKI_IDS is None
    else [w for w in WIKI_CATALOGUE if w["wiki_id"] in WIKI_IDS]
)

print(f"Pulling {len(target_wikis)} wiki(s):")
for w in target_wikis:
    print(f"  {w['wiki_id']:>8}  {w['name']}")

Pulling 2 wiki(s):
   3454593  Dandy's World Wiki
     35171  Hunger Games


## 3. Pull data

In [4]:
def build_wiki_query(wiki_id: int, bucket_prefix: str) -> str:
    post_path  = f"{bucket_prefix}/{wiki_id}/post_chunk_*.parquet"
    rev_path   = f"{bucket_prefix}/{wiki_id}/post_revision_chunk_*.parquet"
    image_path = f"{bucket_prefix}/{wiki_id}/content_image_chunk_*.parquet"
    poll_path  = f"{bucket_prefix}/{wiki_id}/polls_chunk_*.parquet"

    return f"""
    WITH latest_rev AS (
        SELECT
            post_id,
            title,
            raw_content,
            created_at AS revision_created_at,
            id         AS post_revision_id
        FROM read_parquet('{rev_path}')
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY post_id
            ORDER BY created_at DESC, id DESC
        ) = 1
    ),
    image_counts AS (
        SELECT post_revision_id, COUNT(*) AS image_count
        FROM read_parquet('{image_path}')
        GROUP BY post_revision_id
    ),
    poll_counts AS (
        SELECT post_revision_id, COUNT(*) AS poll_count
        FROM read_parquet('{poll_path}')
        GROUP BY post_revision_id
    )
    SELECT
        p.site_id                                       AS wiki_id,
        p.thread_id,
        p.id                                            AS post_id,
        p.position,
        CASE WHEN p.position = 1 THEN 'op' ELSE 'reply' END AS post_role,
        lr.title,
        lr.raw_content                                  AS content,
        p.created_at                                    AS post_time,
        p.created_by                                    AS user_id,
        CAST(NULL AS VARCHAR)                           AS user_name,
        COALESCE(p.upvote_count, 0)                     AS n_likes,
        CASE WHEN COALESCE(img.image_count, 0) > 0 THEN 1 ELSE 0 END AS has_image,
        CASE WHEN COALESCE(pol.poll_count,  0) > 0 THEN 1 ELSE 0 END AS has_poll
    FROM read_parquet('{post_path}') p
    LEFT JOIN latest_rev lr
        ON p.id = lr.post_id
    LEFT JOIN image_counts img
        ON lr.post_revision_id = img.post_revision_id
    LEFT JOIN poll_counts pol
        ON lr.post_revision_id = pol.post_revision_id
    """


frames = []
for w in target_wikis:
    wiki_id = w["wiki_id"]
    print(f"Fetching wiki {wiki_id} ({w['name']}) ...", end=" ", flush=True)
    q = build_wiki_query(wiki_id, BUCKET_PREFIX)
    df = con.sql(q).df()
    df["wiki_name"] = w["name"]
    frames.append(df)
    print(f"{len(df):,} rows")

posts = pd.concat(frames, ignore_index=True)

# Fill user_name deterministically from user_id
posts["user_name"] = posts["user_id"].apply(
    lambda uid: generate_username(user_id=uid) if pd.notna(uid) else generate_username()
)

print(f"\nTotal rows: {len(posts):,}")
posts.head()


Fetching wiki 3454593 (Dandy's World Wiki) ... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2,588,232 rows
Fetching wiki 35171 (Hunger Games) ... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

34,863 rows

Total rows: 2,623,095


,wiki_id,thread_id,post_id,position,post_role,title,content,post_time,user_id,user_name,n_likes,has_image,has_poll,wiki_name
0,3454593,4400000000000280835,4400000000001732375,15,reply,None,"""Nah, I'll just."" - 👻\n",2025-03-17 19:43:21,56366436,KindBear7267,0,1,0,Dandy's World Wiki
1,3454593,4400000000000280815,4400000000001732397,9,reply,None,oof-\n,2025-03-17 19:44:53,53168844,SwiftSage9765,0,1,0,Dandy's World Wiki
2,3454593,4400000000000280848,4400000000001732446,1,op,my love for toons!,,2025-03-17 19:48:26,56673154,DeftRook7412,0,1,0,Dandy's World Wiki
3,3454593,4400000000000280852,4400000000001732476,1,op,Q.,"I put ""Hunting November"" By Adriana M. by the ...",2025-03-17 19:51:12,52134737,WittyEcho9781,2,1,0,Dandy's World Wiki
4,3454593,4400000000000280846,4400000000001732491,10,reply,None,Mirai: *summons the enemy from this picture*\n...,2025-03-17 19:52:51,54504527,BraveEcho5915,0,1,0,Dandy's World Wiki


In [6]:
import re
import sys
import os

# Make utils/ importable from anywhere inside the repo
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.taxonomy import classify_post

# Map the full 19-label taxonomy down to the 9 requested categories
_LABEL_MAP = {
    "trading_or_lfg":           "Trading",
    "fan_creation_or_showcase": "Fanfic",
    "ranking_or_comparison":    "Ranking",
    "opinion_or_reaction":      "Opinion",
    "news_or_update_discussion":"News",
    "theory_or_speculation":    "Theories",
    "question_general":         "Questions",
    "question_how_to":          "Questions",
    "question_lore":            "Questions",
    "question_bug_report":      "Questions",
}

# Event patterns (not in the base taxonomy)
_EVENT = [
    re.compile(r'\b(event|collab|collaboration|limited time|seasonal|holiday|halloween|christmas|anniversary)\b', re.IGNORECASE),
    re.compile(r'\b(contest|competition|tournament|challenge|giveaway|community event)\b', re.IGNORECASE),
    re.compile(r'\b(double xp|bonus|special offer|promo|promotion)\b', re.IGNORECASE),
]

def _is_event(text: str) -> bool:
    return any(p.search(text) for p in _EVENT)

def categorize(row) -> str:
    # Check Poll first (structural signal, not text-based)
    if row.get("has_poll"):
        return "Poll"

    text = " ".join(filter(None, [
        str(row.get("title") or ""),
        str(row.get("content") or ""),
    ])).lower()

    # Check Event (not covered by base taxonomy)
    if _is_event(text):
        return "Event"

    # Build the row dict classify_post expects
    classify_row = {
        "text_lower":   text,
        "title":        row.get("title"),
        "tok":          len(text.split()),
        "poll_count":   int(row.get("has_poll", 0)),
        "image_count":  int(row.get("has_image", 0)),
        "og_count":     0,
        "post_role":    row.get("post_role", "reply"),
        "content_type": (
            "text_plus_media" if (row.get("has_image") or row.get("has_poll")) and text.strip()
            else "media_only"  if (row.get("has_image") or row.get("has_poll"))
            else "text_only"   if text.strip()
            else "empty"
        ),
    }

    label = classify_post(classify_row)
    return _LABEL_MAP.get(label, "General")

posts["category"] = posts.apply(categorize, axis=1)

print(posts["category"].value_counts())
posts[["post_id", "post_role", "title", "category"]].head(10)


category
General      2293404
Opinion       116647
Fanfic         51966
Poll           46001
Questions      37187
Event          23316
Ranking        18249
Theories       16916
Trading        10692
News            8717
Name: count, dtype: int64


,post_id,post_role,title,category
0,4400000000001732375,reply,None,General
1,4400000000001732397,reply,None,General
2,4400000000001732446,op,my love for toons!,General
3,4400000000001732476,op,Q.,General
4,4400000000001732491,reply,None,General
5,4400000000001732492,op,hi this is [player],General
6,4400000000001732498,reply,None,General
7,4400000000001732502,op,doodle page done (GORE WARNING),General
8,4400000000001732505,op,THE BACHELORETTE!! [Meg],Fanfic
9,4400000000001732515,op,CAN I BE UR ONLINE SIBLINGS,General


## 4. Schema & basic stats

In [7]:
print(posts.dtypes)
print()
posts.describe(include="all")

wiki_id               int64
thread_id             int64
post_id               int64
position              int64
post_role            object
title                object
content              object
post_time    datetime64[ns]
user_id               int64
user_name            object
n_likes               int64
has_image             int32
has_poll              int32
wiki_name            object
category             object
dtype: object



,wiki_id,thread_id,post_id,position,post_role,title,content,post_time,user_id,user_name,n_likes,has_image,has_poll,wiki_name,category
count,2.623095e+06,2.623095e+06,2.623095e+06,2.623095e+06,2623095,415160,2623080,2623095,2.623095e+06,2623095,2.623095e+06,2.623095e+06,2.623095e+06,2623095,2623095
unique,NaN,NaN,NaN,NaN,2,364777,2155669,NaN,NaN,9226,NaN,NaN,NaN,2,10
top,NaN,NaN,NaN,NaN,reply,Repost,,NaN,NaN,FaintApex7159,NaN,NaN,NaN,Dandy's World Wiki,General
freq,NaN,NaN,NaN,NaN,2207071,1737,185179,NaN,NaN,63709,NaN,NaN,NaN,2588232,2293404
mean,3.409146e+06,4.399998e+18,4.400000e+18,1.945025e+02,NaN,NaN,NaN,2025-09-08 17:07:57.260818688,5.572693e+07,NaN,5.551324e-01,1.355204e-01,1.753692e-02,NaN,NaN
min,3.517100e+04,3.343173e+18,4.400000e+18,1.000000e+00,NaN,NaN,NaN,2025-03-06 00:03:10,8.000000e+00,NaN,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN
25%,3.454593e+06,4.400000e+18,4.400000e+18,2.000000e+00,NaN,NaN,NaN,2025-05-22 15:04:59,5.542713e+07,NaN,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN
50%,3.454593e+06,4.400000e+18,4.400000e+18,6.000000e+00,NaN,NaN,NaN,2025-09-18 04:31:13,5.642998e+07,NaN,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN
75%,3.454593e+06,4.400000e+18,4.400000e+18,1.700000e+01,NaN,NaN,NaN,2025-12-13 23:05:19,5.714223e+07,NaN,1.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN
max,3.454593e+06,4.400000e+18,4.400000e+18,3.389900e+04,NaN,NaN,NaN,2026-03-09 20:44:10,6.011979e+07,NaN,5.580000e+02,1.000000e+00,1.000000e+00,NaN,NaN


In [8]:
# Row counts per wiki
posts.groupby("wiki_id").size().rename("posts").reset_index().sort_values("posts", ascending=False)

,wiki_id,posts
1,3454593,2588232
0,35171,34863


In [9]:
# Null rates
null_pct = (posts.isnull().mean() * 100).round(2).rename("null_%")
null_pct

wiki_id       0.00
thread_id     0.00
post_id       0.00
position      0.00
post_role     0.00
title        84.17
content       0.00
post_time     0.00
user_id       0.00
user_name     0.00
n_likes       0.00
has_image     0.00
has_poll      0.00
wiki_name     0.00
category      0.00
Name: null_%, dtype: float64

## 5. (Optional) Export

In [ ]:
import json
from pathlib import Path

def export_sample(
    df: pd.DataFrame,
    n: int = 200,
    output_path: str = "results/sample_export.json",
    stratify_by: str | list[str] | None = "category",
    n_per_group: int | None = None,
    random_seed: int = 42,
) -> str:
    """
    Randomly sample rows from df and export as a JSON file.

    Parameters
    ----------
    df            : source DataFrame
    n             : total rows to sample when n_per_group is None (default 200)
    output_path   : destination file path
    stratify_by   : column name or list of column names to stratify on.
                    When a list is given (e.g. ["wiki_id", "category"]), rows are
                    distributed equally across every (wiki_id × category) combination.
                    When n_per_group is None, allocation is equal across groups.
    n_per_group   : if set, take exactly this many rows from EACH unique group
                    combination (ignores n).
    random_seed   : for reproducibility

    Returns
    -------
    Absolute path of the written file.
    """
    # Normalise stratify_by to a list
    if isinstance(stratify_by, str):
        stratify_cols = [stratify_by]
    elif stratify_by:
        stratify_cols = list(stratify_by)
    else:
        stratify_cols = []

    valid_cols = [c for c in stratify_cols if c in df.columns]

    if n_per_group is not None and valid_cols:
        parts = []
        for _keys, subset in df.groupby(valid_cols):
            k = min(n_per_group, len(subset))
            parts.append(subset.sample(k, random_state=random_seed))
        sample = pd.concat(parts).sample(frac=1, random_state=random_seed)

    elif valid_cols:
        # Equal allocation across every unique combination of stratify columns
        total = min(n, len(df))
        groups = list(df.groupby(valid_cols))
        n_groups = len(groups)
        base = total // n_groups
        remainder = total - base * n_groups

        parts = []
        for i, (_keys, subset) in enumerate(groups):
            quota = base + (1 if i < remainder else 0)
            k = min(quota, len(subset))
            if k > 0:
                parts.append(subset.sample(k, random_state=random_seed))
        sample = pd.concat(parts).sample(frac=1, random_state=random_seed)

    else:
        sample = df.sample(min(n, len(df)), random_state=random_seed)

    # Serialise: convert timestamps and NaNs to JSON-safe types
    records = (
        sample
        .assign(**{
            col: sample[col].astype(str)
            for col in sample.select_dtypes(include=["datetime64[ns, UTC]", "datetime64[ns]"]).columns
        })
        .where(sample.notna(), other=None)
        .to_dict(orient="records")
    )

    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2, default=str)

    print(f"Exported {len(records)} rows → {out.resolve()}")
    if valid_cols:
        dist = sample.groupby(valid_cols).size().rename("count")
        print(dist.to_string())
    return str(out.resolve())


# 200 rows evenly distributed across (wiki_id × category) combinations
export_sample(posts, n=200, stratify_by=["wiki_id", "category"], output_path="results/sample_export.json")


Exported 200 rows → /Users/jdeng/Engineering/claude-work/flair_tagging/results/sample_export.json
category
Event        20
Fanfic       20
General      20
News         20
Opinion      20
Poll         20
Questions    20
Ranking      20
Theories     20
Trading      20


'/Users/jdeng/Engineering/claude-work/flair_tagging/results/sample_export.json'

In [11]:
posts.dtypes

wiki_id               int64
thread_id             int64
post_id               int64
position              int64
post_role            object
title                object
content              object
post_time    datetime64[ns]
user_id               int64
user_name            object
n_likes               int64
has_image             int32
has_poll              int32
wiki_name            object
category             object
dtype: object